In [ ]:
library(data.table)
library(dplyr, warn.conflicts = FALSE)
library(zoo, warn.conflicts = FALSE)
library(glue, warn.conflicts = FALSE)
library(RPostgres) 
library(arrow)

In [ ]:
format_time_period <- function(df, time_period){
    
    df <- df %>%
      mutate(PERIOD = as.character(PERIOD)) %>%
      mutate(Agg_period = as.yearmon(PERIOD, format='%Y%m')) %>%
      mutate(Annee = as.numeric(substring(Agg_period, 4, 8))) %>%
      mutate(Filtre_time = 'Mensuel') %>%
      mutate(Agg_period = as.character(Agg_period)) # Yearmon throwing strange errors
      
    if (time_period == 'trimestre') {
        df = df %>%
             mutate(Agg_period = case_when(
                                     grepl("Jan|Feb|Mar", PERIOD) ~ 'T1',
                                     grepl("Apr|May|Jun", PERIOD) ~ 'T2',
                                     grepl("Jul|Aug|Sep", PERIOD) ~ 'T3',
                                     grepl("Oct|Nov|Dec", PERIOD) ~ 'T4')) %>%
             mutate(Filtre_time = 'Trimestre')
    } else if(time_period == 'semestre') {
        df = df %>%
             mutate(Agg_period = case_when(grepl("Jan|Feb|Mar|Apr|May|Jun", 
                                                 PERIOD) ~ 'S1',
                                           grepl("Jul|Aug|Sep|Oct|Nov|Dec", 
                                                 PERIOD) ~ 'S2')) %>%
             mutate(Filtre_time = 'Semestre')


    } else if(time_period == 'annee') {
        df = df %>% 
             mutate(Agg_period = Annee) %>%
             mutate(Filtre_time = 'Année')
    }
        
    if(time_period %in% c('semestre', 'trimestre')){
        df <- df %>% mutate(Agg_period = paste(Agg_period, Annee))
    }
        
   return(df)     
}

Read data

In [ ]:
# Parameters
ANNEE_A_ANALYSER = 2024
UPLOAD = FALSE

In [ ]:
# Paths
DATA_PATH = '/home/hexa/workspace/pnlp-tdb-pipeline/data/'
INTERIM_DATA_PATH = paste0(DATA_PATH, 'interim/availability/')
OUTPUT_DATA_PATH = paste0(DATA_PATH, 'processed/')

In [ ]:
# if(ANNEE_A_ANALYSER %in% c(2018, 2019)){
#     path_stub <- paste0(DATA_PATH,
#                         "indicators_availability_level_{geo}_",
#                         "month_newformat_2018-2019.csv")
# } else {
#     path_stub <- paste0(DATA_PATH,
#                         "indicators_availability_level_{geo}_",
#                         "month_newformat_{ANNEE_A_ANALYSER}.csv")
# }

path_stub <- paste0(INTERIM_DATA_PATH,
                    "indicators_availability_level_{geo}_",
                    "month_newformat_{ANNEE_A_ANALYSER}.feather")


In [ ]:
geo <- 4
aires <- read_feather(glue(path_stub))

# if(ANNEE_A_ANALYSER %in% c(2018, 2019)){
#     if(ANNEE_A_ANALYSER == 2018){
#         aires <- aires[PERIOD < 201901]
#     }
#     else {
#         aires <- aires[PERIOD > 201812]
#     }
# }

aires <- format_time_period(aires, 'mois')

In [ ]:
# aires de santé
setDT(aires)
aires[, LEVEL_1_NAME := 'République Démocratique du Congo']

In [ ]:
# s3write_using(x = aires,
#               FUN = fwrite,
#               object = paste0(OUTPUT_PATH,
#                               "PNLP_dash_availability_",
#                               ANNEE_A_ANALYSER,
#                               "_ND.csv"))

write_feather(aires, 
              paste0(OUTPUT_DATA_PATH, 
                    "PNLP_dash_availability_", 
                    ANNEE_A_ANALYSER,
                    ".feather"))

Write to DB

In [ ]:
con <- dbConnect(
    RPostgres::Postgres(),
    dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
    host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
    port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
    user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
    password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
)

In [ ]:
if (UPLOAD) {
#     df_22 <- s3read_using(FUN = fread, 
#                           object = paste0(OUTPUT_PATH, 
#                                           "PNLP_dash_availability_2022_ND.csv"))
    
#     df_21 <- s3read_using(FUN = fread, 
#                           object = paste0(OUTPUT_PATH, 
#                                           "PNLP_dash_availability_2021_ND.csv"))

#     df_20 <- s3read_using(FUN = fread, 
#                           object = paste0(OUTPUT_PATH, 
#                                           "PNLP_dash_availability_2020_ND.csv"))

#     df_19 <- s3read_using(FUN = fread, 
#                           object = paste0(OUTPUT_PATH, 
#                                           "PNLP_dash_availability_2019_ND.csv"))

#     df_18 <- s3read_using(FUN = fread,
#                           object = paste0(OUTPUT_PATH, 
#                                           "PNLP_dash_availability_2018_ND.csv"))
    
#     output_combined <- do.call(rbind, list(df_18, df_19, df_20, df_21, df_22))
    
    file.names <- list.files(OUTPUT_DATA_PATH, pattern = 'availability.+feather')
    file.paths <- paste0(OUTPUT_DATA_PATH, file.names)
    
    output.data <- lapply(file.paths, read_feather)
    output.data <- lapply(output.data, setDT)
    
    output.combined <- rbindlist(output.data)
    
    # fix names (@Fernando)
    output.combined[, ID_NAME := gsub('Ã©', 'é', 
                                      ID_NAME)]
    output.combined[, ID_NAME := gsub('dâ€™insecticide', "d'Insecticide", 
                                      ID_NAME)]
    
    dbWriteTable(con, "availability", output.combined, overwrite=TRUE)
}


**The following code was used to revert the DB table to december 2024**

In [ ]:
# file.names <- c("PNLP_dash_availability_2018.feather", "PNLP_dash_availability_2019.feather", 
#                 "PNLP_dash_availability_2020.feather", "PNLP_dash_availability_2021.feather", 
#                 "PNLP_dash_availability_2022.feather", "PNLP_dash_availability_2023.feather",
#                 "PNLP_dash_availability_2024.feather")

# file.paths <- paste0(OUTPUT_DATA_PATH, file.names)

# output.data <- lapply(file.paths, read_feather)
# output.data <- lapply(output.data, setDT)
# output.combined <- rbindlist(output.data)

# # fix names (@Fernando)
# output.combined[, ID_NAME := gsub('Ã©', 'é', ID_NAME)]
# output.combined[, ID_NAME := gsub('dâ€™insecticide', "d'Insecticide", ID_NAME)]

In [ ]:
# con <- dbConnect(
#     RPostgres::Postgres(),
#     dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
#     host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
#     port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
#     user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
#     password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
# )
# dbWriteTable(con, "availability", output.combined, overwrite=TRUE)